# Phishing Website Detection — Step 2: Your First Baseline Model

**Capstone roadmap:**
1. ~~Dataset acquisition & exploration~~ ✅
2. **Step 2 (this notebook): baseline model + proper evaluation**
3. Model improvement
4. Wrap model in an API
5. Build the browser extension

### What we confirmed in Step 1
`Result = -1` → phishing, `Result = 1` → legitimate. We'll use that here.

### What this notebook teaches
- Why we split data into training and test sets (and never evaluate on training data)
- Training a simple, interpretable model first: **Logistic Regression**
- Why **accuracy alone is misleading** for this problem, and what to use instead
- Reading a **confusion matrix** and understanding false positives vs. false negatives *in the context of phishing detection specifically*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid")

## 1. Load the data we saved in Step 1

Make sure `phishing_data.csv` is in the same folder as this notebook (or update the path below).

In [ ]:
df = pd.read_csv('phishing_data.csv')
print(df.shape)
df.head()

## 2. Separate features (X) from the label (y)

`X` is everything the model is *allowed to look at* to make a decision. `y` is what it's trying to predict. We must never let `y` leak into `X`.

In [ ]:
X = df.drop(columns=['Result'])
y = df['Result']

print("X shape:", X.shape)
print("y shape:", y.shape)

## 3. Train/test split — the most important habit in ML

**Why do we split the data at all?**

If we train a model and then test it on the *same* data it learned from, it can just memorize the answers — like grading a student on the exact homework questions they already saw the answers to. That tells us nothing about how well it will handle a phishing site it's never seen before, which is the entire point of this project.

So we split the data:
- **Training set (typically 70–80%)** — the model learns patterns from this
- **Test set (the remaining 20–30%)** — held back completely, used only at the end to check performance

We use `stratify=y` to make sure both sets keep the same phishing/legitimate ratio as the full dataset — otherwise, by random chance, our test set could end up with way more (or fewer) phishing examples than it should.

`random_state=42` just makes the split reproducible — you'll get the exact same split every time you run this cell (42 is a common convention, no special meaning).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("\nClass balance in training set:")
print(y_train.value_counts(normalize=True))
print("\nClass balance in test set:")
print(y_test.value_counts(normalize=True))

Notice the two class balances above are nearly identical — that's `stratify=y` doing its job.

## 4. Train a baseline model: Logistic Regression

**Why start here instead of something more powerful like Random Forest or XGBoost?**

Logistic Regression is simple, fast, and interpretable — it essentially learns a weight for each feature indicating how much it pushes the prediction toward "phishing" or "legitimate." Starting simple gives us a **baseline number**. Later, when we try fancier models, we'll know exactly how much they actually improve things — instead of just assuming a complex model is automatically better (often it isn't, or the gain isn't worth the added complexity).

`max_iter=1000` just gives the optimizer enough iterations to converge on this dataset size — you may see a convergence warning with the default value.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

## 5. Make predictions on the test set

This is the model predicting on data it has never seen during training — the real test.

In [ ]:
y_pred = model.predict(X_test)
y_pred[:10]  # first 10 predictions, just to peek

## 6. Evaluate — and why accuracy alone isn't enough

**Accuracy** = (correct predictions) / (total predictions). Simple, but dangerously incomplete here.

Imagine a lazy model that *always* predicts "legitimate," no matter the input. If our test set is, say, 56% legitimate sites, that lazy model scores 56% accuracy while catching **zero** phishing sites — completely useless, but the accuracy number alone doesn't reveal that.

For phishing detection specifically, we care about two other numbers:

- **Recall** (for the phishing class): out of all *actual* phishing sites, how many did we catch? Low recall = phishing sites slipping through undetected. **This is usually the most important number for this problem** — missing a real phishing site is the costly mistake.
- **Precision** (for the phishing class): out of all sites we *flagged* as phishing, how many really were? Low precision = we're crying wolf on legitimate sites, annoying real users.
- **F1 score**: a single number balancing precision and recall.

There's a real tradeoff between recall and precision — tuning a model to catch more phishing sites (higher recall) usually means flagging a few more legitimate sites by mistake (lower precision), and vice versa. Which one matters more depends on the product: a strict corporate email filter might tolerate more false alarms to catch more real phishing, while a consumer browser extension might avoid annoying users with false alarms. Worth thinking about as you interpret these numbers.

In [ ]:
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision (phishing=-1):", precision_score(y_test, y_pred, pos_label=-1))
print("Recall (phishing=-1):   ", recall_score(y_test, y_pred, pos_label=-1))
print("F1 score (phishing=-1): ", f1_score(y_test, y_pred, pos_label=-1))

## 7. Confusion matrix — see exactly *where* the model gets it wrong

A confusion matrix breaks predictions into 4 buckets:

|  | Predicted phishing | Predicted legitimate |
|---|---|---|
| **Actually phishing** | ✅ True Positive | ❌ False Negative — **dangerous**: a real phishing site was let through |
| **Actually legitimate** | ⚠️ False Positive — a real site got wrongly flagged | ✅ True Negative |

The **False Negative** cell is the one to watch most closely for this project — it's a phishing site that slipped past the model undetected.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: Phishing', 'Predicted: Legitimate'],
    yticklabels=['Actual: Phishing', 'Actual: Legitimate']
)
plt.title('Confusion Matrix — Logistic Regression baseline')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# A full summary report combining precision, recall, and F1 for both classes at once
print(classification_report(y_test, y_pred, target_names=['Phishing (-1)', 'Legitimate (1)']))

## 8. Which features mattered most to this model?

Logistic Regression assigns a **coefficient** (weight) to each feature. A large positive coefficient pushes the prediction toward `1` (legitimate); a large negative coefficient pushes it toward `-1` (phishing). Looking at these is a nice sanity check — do the most influential features match what makes intuitive sense (e.g. domain age, SSL status)?

In [ ]:
coefficients = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 6))
coefficients.head(10).plot(kind='barh')
plt.title('Top 10 most influential features (Logistic Regression)')
plt.xlabel('Coefficient (negative → pushes toward phishing)')
plt.gca().invert_yaxis()
plt.show()

## Summary — what we now have

- [ ] A trained baseline model with real, held-out-test-set numbers (not just training accuracy)
- [ ] An understanding of precision vs. recall vs. F1, and *why* recall on the phishing class matters most here
- [ ] A confusion matrix showing exactly how many phishing sites slipped through undetected
- [ ] A sense of which features actually drive the prediction

**Your task before Step 3:** write down (edit this cell) your baseline numbers here, so we can compare against them later:

- Accuracy: ___
- Precision (phishing): ___
- Recall (phishing): ___
- F1 (phishing): ___

**Next up (Step 3):** we'll try more powerful models (Random Forest, and maybe Gradient Boosting), compare them fairly against this baseline, and pick a final model — plus save it to disk so the API in Step 4 can load and use it.